### Численные методы многомерной минимизации c использованием производных первого порядка
Методы доверительной области:
1. метод доверительной области с локально оптимальным шагом
2. метод доверительной области с изломом
Многошаговые методы:
3. метод тяжёлого шарика
4. метод ускоренного градиентного спуска Нестерова
5. метод Флетчера-Ривза
6. метод Полака-Рибьера
Квазиньютоновские методы:
7. метод Давидона-Флетчера-Пауэлла
8. метод Бройдена
9. метод с симметричной формулой ранга 1
10. метод Бройдена-Флетчера-Гольдфарба-Шанно

Вариант 13
#### метод доверительной области с локально оптимальным шагом

#### метод с симметричной формулой ранга 1
|функция|
| --- |
|f(x1,x2)=100(1-x1^2-x2^2)^2+(x1-1)^2+x2^2|

Для градиентных методов исследовать сходимость алгоритма, фиксируя найденное решение, значение функции, количество итераций метода и количество вычислений функции в зависимости от задаваемой точности поиска (ε = 10−2, 10−4, 10−6). Результаты оформить в виде таблицы:

Для метода второго порядка исследовать сходимость алгоритма по числу итераций при выборе различных начальных точек (не менее трёх). Результаты оформить в виде таблицы:

In [ ]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go
from scipy.optimize import minimize

def Cauchy(g, B, delta):
    norm_g = np.linalg.norm(g)
    if norm_g == 0:
        return np.zeros_like(g)
    
    p_s = - (delta / norm_g) * g

    gBg = g.T @ B @ g
    if gBg <= 0:
        tau = 1.0
    else:
        tau = min(norm_g**3 / (delta * gBg), 1.0)

    return tau * p_s

def trust_region_method(f, x0, delta0=1.0, delta_hat=50.0, eta=0.1, max_iter=1000, tol=1e-6):
    x = x0.astype(float)
    x1, x2 = sp.symbols('x1, x2')
    delta = delta0
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0.copy()]
    values = [fun(x[0], x[1])]
    k = 0

    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    hessian = sp.hessian(f, [x1, x2])
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")
    
    for i in range(max_iter):
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < tol:
            break
        B = np.array(hessian.subs({x1: x[0], x2: x[1]})).astype(float) #Заменить на приближение
        
        p = Cauchy(c_grad, B, delta)
        norm_p = np.linalg.norm(p)

        f_x = fun(x[0], x[1])
        f_xp = fun(x[0] + p[0], x[1] + p[1])
        mp = c_grad @ p + 0.5 * (p.T @ B @ p)
        actual_reduction = f_x - f_xp
        predicted_reduction = -mp 

        if predicted_reduction == 0:
            rho = 0.0
        else:
            rho = actual_reduction / predicted_reduction

        if rho < 0.25:
            delta = 0.25 * delta
        elif rho > 0.75:
            delta = min(2.0 * delta, delta_hat)

        if rho > eta:
            x = x + p
            trajectory.append(x.copy())
            values.append(fun(x[0], x[1]))
        k+=1

    return np.array(trajectory), np.array(values), k




x1, x2 = sp.symbols('x1, x2')
f = 100*(1 - x1**2 - x2**2)**2 + (x1-1)**2 + x2**2
np.random.seed(42)
p_list = (np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2))

x1_range = np.linspace(-10, 10, 100)
x2_range = np.linspace(-10, 10, 100)
x, y = np.meshgrid(x1_range, x2_range)
fu = 100*(1 - x**2 - y**2)**2 + (x-1)**2 + y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'100(1-x1^2-x2^2)^2+(x1-1)^2+x2^2') 

# method visualize

for point in p_list:
    print(f'point = {point}')
    ans = trust_region_method(f, point, delta0=10, eta=0.1)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point.tolist()}'))
    fig.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig.show()

point = [-0.75275929  2.70428584]
[1.00000000e+00 4.67390735e-07] 2.1848749049487794e-13 742
point = [1.39196365 0.59195091]
[1.00000000e+00 4.36181393e-07] 1.9026250356164884e-13 221
point = [-2.06388816 -2.06403288]
[ 0.99999709 -0.00149353] 2.231937113156644e-06 1000


In [ ]:
#Вариант приближённого вычисления
import numpy as np
from scipy.optimize import minimize

# Целевая функция (например, функция Розенброка)
def rosen(x):
    return sum(100.0*(x[1:]-x[:-1]**2.0)**2.0 + (1.0-x[:-1])**2.0)

def fu(x):
    return 100*(1 - x[0]**2 - x[1]**2)**2 + (x[0]-1)**2 + x[1]**2

x0 = np.array([0.0, 0.0])
# Используем метод BFGS
res = minimize(fu, x0, method='BFGS')

# Приближенный гессиан (inverse Hessian)
print(res.hess_inv)